# Practical 3: Conversation Memory in an AI Agent (Google Gemini API — Colab)

**Aim:** Implement conversation memory in an AI agent so that it can remember previous interactions and generate contextual responses, using the Google `genai` SDK directly (no LangChain).

**Steps covered:**
1. Install required libraries
2. Import required modules
3. Initialize the Gemini client
4. Create conversation memory
5. Build the conversation agent (memory + model)
6. Start a conversation
7. Continue the conversation
8. Test contextual responses
9. Display conversation memory
10. Analyze results (with vs. without memory)


## Step 1: Install Required Libraries

Run this cell first in Colab.

In [ ]:
!pip install -q -U google-genai

## Step 2: Import Required Modules

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata

## Step 3: Initialize the Gemini Client

**Get a free Gemini API key from:** https://aistudio.google.com/app/apikey

**Recommended (secure) way in Colab:**
1. Click the key icon in the left sidebar of Colab.
2. Add a new secret named `GEMINI_API_KEY` and paste your key as the value.
3. Toggle "Notebook access" on for it.
4. Run the cell below — it reads the key from Colab secrets automatically.

If you don't want to use secrets, you can instead pass the key directly as
shown in the commented line.

In [ ]:
# --- OPTION A (recommended): read key from Colab Secrets ---
API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=API_KEY)

# --- OPTION B: hardcode directly (quick, not recommended for shared notebooks) ---
# client = genai.Client(api_key="YOUR_GEMINI_API_KEY")

# Quick sanity check that the client/model is working
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="What is Agentic AI?"
)
print(response.text)

## Step 4: Create Conversation Memory

The `genai` SDK does not include a built-in memory object like LangChain does,
so we build one ourselves: a simple Python list that stores every user message
and model response as `types.Content` objects. This list IS the agent's memory —
we will pass it back into `contents` on every call so the model can "see" the
full conversation so far.

In [ ]:
conversation_memory = []  # this list holds the entire chat history

def add_to_memory(role, text):
    """Append a message to conversation memory.
    role must be either 'user' or 'model'."""
    conversation_memory.append(
        types.Content(role=role, parts=[types.Part(text=text)])
    )

## Step 5: Build the Conversation Agent

This function connects the Gemini model with our memory list: it adds the new
user message to memory, sends the *entire* memory as `contents` so the model
has full context, then stores the model's reply back into memory too.

In [ ]:
def chat(user_input, model="gemini-2.5-flash"):
    # 1. Add the user's new message to memory
    add_to_memory("user", user_input)

    # 2. Send the FULL conversation history (memory) to the model
    response = client.models.generate_content(
        model=model,
        contents=conversation_memory
    )

    # 3. Store the model's reply in memory too
    add_to_memory("model", response.text)

    return response.text

## Step 6: Start a Conversation

Ask an initial question and verify the agent responds correctly.

In [ ]:
reply1 = chat("Hi, my name is Rahul and I'm learning about AI agents.")
print("Agent:", reply1)

## Step 7: Continue the Conversation

Ask follow-up questions related to the previous interaction and observe whether
the agent remembers earlier responses.

In [ ]:
reply2 = chat("What's a good first project to build as a beginner?")
print("Agent:", reply2)

In [ ]:
reply3 = chat("Can you explain that in simpler terms?")
print("Agent:", reply3)

## Step 8: Test Contextual Responses

This question can only be answered correctly if the agent is actually using
stored memory rather than treating each query independently.

In [ ]:
reply4 = chat("What's my name, and what was the project you suggested earlier?")
print("Agent:", reply4)

## Step 9: Display Conversation Memory

Retrieve and display the complete conversation history stored in memory to
confirm all user queries and agent responses were recorded.

In [ ]:
for i, turn in enumerate(conversation_memory):
    role = turn.role
    text = turn.parts[0].text
    print(f"[{i}] {role.upper()}: {text}\n")

## Step 10: Analyze the Results — With vs. Without Memory

Run the same follow-up question through a **memory-less** call (no history
passed in) for comparison.

In [ ]:
# No-memory baseline: a fresh call to the model with no prior context
no_memory_response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="What's my name, and what was the project you suggested earlier?"
)
print("Without memory:", no_memory_response.text)

### Comparison

| Aspect | Without Memory | With Memory |
|---|---|---|
| Recalls user's name | No | Yes |
| Recalls earlier suggestion | No | Yes |
| Response coherence across turns | Low — each query isolated | High — builds on prior context |
| Use case fit | One-off Q&A | Multi-turn assistants, chatbots |

**Conclusion:** Conversation memory transforms a stateless LLM call into a
context-aware agent by persisting prior turns and re-sending them with every
new request. This is essential for chatbots, tutoring agents, and any assistant
expected to hold a coherent multi-turn dialogue.


## Bonus: Using the SDK's Built-in Chat Session

The `google-genai` SDK also offers a `client.chats.create()` helper that
manages conversation memory for you automatically, so you don't have to
maintain the `conversation_memory` list by hand.

In [ ]:
chat_session = client.chats.create(model="gemini-2.5-flash")

r1 = chat_session.send_message("Hi, my name is Rahul.")
print("Agent:", r1.text)

r2 = chat_session.send_message("What's my name?")
print("Agent:", r2.text)

# The session keeps its own history internally:
for msg in chat_session.get_history():
    print(msg.role, ":", msg.parts[0].text)